In [ ]:
#VAR model - bi-directional forecasting model with multiple time series influencing each other (predictor variables influence target variable and vice versa)
import statsmodels
import pandas as pd
from pandas import read_csv
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import math

from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller
from statsmodels.tools.eval_measures import rmse, aic, bic, hqic

#from sklearn.metrics import mean_squared_error
#AIC - Akaike Information Criteria
#BIC - Schwarz-Bayesian Information Criteria
#HQIC - Hannan-Quinn Information Criteria
#FPE - Focused Prediction Error

In [ ]:
#Steps for VAR model
#1. Load and Examine/Visualize the data
#2. Cointegration test
#3. Test for data stationarity (ADF)
#4. If non-stationary, take the difference(s)
#5. Test for data stationarity again
#6. Granger's causality test to check whether a variable is helpful for forecasting the behaviour of another variable, hence included in a VAR model
#7. Train/Test Split, Find the order P value based on Information Criteria (e.g. AIC etc) for VAR model using train differenced data
#8. Apply the VAR model with order P
#9. Check for impulse responses and FEVD
#10. Check for fitted_model's Serial Correlation of Residuals(Errors) using Durbin Watson (DW) Statistic
#11. Forecast using VAR model
#12. If necessary, invert the earlier transformation and get real forecasted values for all predictors
#13. Plot and calculate accuracy of forecast vs actual values
#14. Evaluate the model performance based on forecast vs actual values based on various metrics (e.g. RMSE etc)
#15. Monte Carlo Simulation for variables m=1000 paths
#16. Given 1 variable & n future timestamps values, try to predict values for all remaining varibles in their n future timestamps based on sorting points


In [ ]:
# load the dataset
series = read_csv('Combined_usdcny_1month_10features.csv', parse_dates=['Date'], index_col='Date')

print(series.dtypes)

print(series.shape)

# Describe columns
series.describe(include='all')

In [ ]:
series.head()

In [ ]:
# Remove not needed columns - only keeping percentage data columns and index data columns
columns_to_remove = ['US Debt:Gov', 'US Export Price Index', 'US M2 Amount', 'CN M2 Amount']
series.drop(columns_to_remove, axis=1, inplace=True)

series.head()

In [ ]:
#visulise the time series
fig, axes = plt.subplots(nrows=3, ncols=2, dpi=200, figsize=(10,10))

for i, ax in enumerate(axes.flatten()):
    data = series[series.columns[i]]
    ax.plot(data, color='red', linewidth=1)
    #set title etc
    ax.set_title(series.columns[i])
    ax.xaxis.set_ticks_position('none')
    ax.yaxis.set_ticks_position('none')
    ax.spines['top'].set_alpha(0)
    ax.tick_params(labelsize=6)
    
plt.tight_layout()

In [ ]:
#Causality test using Granger's causality test - null hypothesis that coefficients of past values in the regression equation is zero; 
#if p-value obtained from the test is less than significance level of 0.05, then reject the null hypothesis, meaning X series causing Y change.
#The test is used to find short-run relationship between variables.

from statsmodels.tsa.stattools import grangercausalitytests

maxlag=4

test='ssr_chi2test'

def grangers_causation_matrix(data, variables, test='ssr_chi2test', verbose=False):
    ##check Granger Causality of all possible combinations of the time series####
    df = pd.DataFrame(np.zeros((len(variables), len(variables))), columns=variables, index=variables)
    for c in df.columns:
        for r in df.index:
            test_result = grangercausalitytests(data[[r,c]], maxlag=maxlag, verbose=False)
            p_values = [round(test_result[i+1][0][test][1],4) for i in range(maxlag) ]
            if verbose: print(f'Y={r}, X={c}, P Values={p_values}')
            min_p_value = np.min(p_values)
            df.loc[r,c] = min_p_value
    df.columns = [var + '_x' for var in variables]
    df.index = [var + '_y' for var in variables]
    return df

grangers_causation_matrix(series, variables=series.columns)

#from below table you can see FX (ccy) rate Y is significantly related to FRD_x and RJ/CRB Index_x with 2 lags(order p)

In [ ]:
#Conintegration Test - when two or more time series, there exists a linear combination of them that has an order of integration (d) less than that of the individual series, then the collection of series is said to be cointegrated.
#Order of integration (d) is nothing but the number of differencing required to make a non-stationary time series stationary.
#This test is used to find relationship in the long-run between variables; check if they are cointegrated, if not then use VAR on the first difference of the variable.
from statsmodels.tsa.vector_ar.vecm import coint_johansen

def cointegration_test(df, alpha=0.05):
    #Perform Johansen's Cointegration Test and output summary
    out = coint_johansen(df, -1, 5)
    d = {'0.90':0, '0.95':1, '0.99':2}
    traces = out.lr1
    cvts = out.cvt[:, d[str(1-alpha)]]
    
    def adjust(val, length=6): return str(val).ljust(length)
    
    #Summary
    print('Name :: Test Stat > C(95%)  => Signif \n', '--'*20)
    for col, trace, cvt in zip(df.columns, traces, cvts):
        print(adjust(col), ':: ', adjust(round(trace,2), 9), '>', adjust(cvt, 8), ' => ', trace > cvt)
        
cointegration_test(series)

In [ ]:
#Check and make time series stationary/mean reversion (ADF test) for VAR model - significance level set to 0.05 by default
#test for a unit root in each component series individually, using univariate unit root tests, e.g. ADF, KPSS, etc
def adfuller_test(series, signif=0.05, name='', verbose=False):
    """Perform ADFuller to test for Stationarity of given series and print report"""
    r = adfuller(series, autolag='AIC')
    output = {'test_statistic':round(r[0], 4), 'pvalue':round(r[1], 4), 'n_lags':round(r[2], 4), 'n_obs':r[3]}
    p_value = output['pvalue'] 
    def adjust(val, length= 6): return str(val).ljust(length)

    # Print Summary
    print(f'    Augmented Dickey-Fuller Test on "{name}"', "\n   ", '-'*47)
    print(f' Null Hypothesis: Data has unit root. Non-Stationary.')
    print(f' Significance Level    = {signif}')
    print(f' Test Statistic        = {output["test_statistic"]}')
    print(f' No. Lags Chosen       = {output["n_lags"]}')

    for key,val in r[4].items():
        print(f' Critical value {adjust(key)} = {round(val, 3)}')

    if p_value <= signif:
        print(f" => P-Value = {p_value}. Rejecting Null Hypothesis.")
        print(f" => Series is Stationary.")
    else:
        print(f" => P-Value = {p_value}. Weak evidence to reject the Null Hypothesis.")
        print(f" => Series is Non-Stationary.")

In [ ]:
#ADF test on each column
for name, column in series.iteritems():
    adfuller_test(column, name=column.name)
    print('\n')

In [ ]:
#split stationary data and non-stationary data
stat = series.drop(["1M USDCNY", "Import Price Index(HS2)"], 1)
nonstat1 = series["1M USDCNY"]
nonstat2 = series["Import Price Index(HS2)"]

In [ ]:
#differencing once to try making time series stationary
#df_differenced = series.diff().dropna()  ---- this differences whole data series
df_differenced = stat.diff().dropna()

df_differenced

In [ ]:
nonstat1_1 = nonstat1[1:]

nonstat2_1 = nonstat2[1:]

In [ ]:
df_differenced1 = pd.concat([nonstat1_1, df_differenced, nonstat2_1], axis=1, join='inner')


In [ ]:
df_differenced1

In [ ]:
#ADF test again on each column
for name, column in df_differenced1.iteritems():
    adfuller_test(column, name=column.name)
    print('\n')

In [ ]:
#Causality test using Granger's causality test - null hypothesis that coefficients of past values in the regression equation is zero; 
#if p-value obtained from the test is less than significance level of 0.05, then reject the null hypothesis, meaning X series causing Y change.

from statsmodels.tsa.stattools import grangercausalitytests

maxlag=4

test='ssr_chi2test'

def grangers_causation_matrix(data, variables, test='ssr_chi2test', verbose=False):
    ##check Granger Causality of all possible combinations of the time series####
    df = pd.DataFrame(np.zeros((len(variables), len(variables))), columns=variables, index=variables)
    for c in df.columns:
        for r in df.index:
            test_result = grangercausalitytests(data[[r,c]], maxlag=maxlag, verbose=False)
            p_values = [round(test_result[i+1][0][test][1],4) for i in range(maxlag) ]
            if verbose: print(f'Y={r}, X={c}, P Values={p_values}')
            min_p_value = np.min(p_values)
            df.loc[r,c] = min_p_value
    df.columns = [var + '_x' for var in variables]
    df.index = [var + '_y' for var in variables]
    return df

grangers_causation_matrix(df_differenced1, variables=series.columns)

#from below table you can see FX (ccy) rate Y is significantly related to FRD_x with max 4 lags(order p)

In [ ]:
#Split the dataset into training and test data. VAR model will be fitted on df_train then used to forecast the next 4 observations, which will be compared against the actual values in test data
sp = 4
df_train, df_test = series[:-sp], series[-sp:]

print(df_train.shape)
print(df_test.shape)


df_train1, df_test1 = df_differenced1[:-sp], df_differenced1[-sp:]

In [ ]:
#select the best order (p) of VAR model
#pick the order that gives a model with least AIC, BIC, FPE, HQIC
model = VAR(df_train1)

for i in [0,1,2,3,4]:
    result = model.fit(i)
    print('Lag Order= ', i)
    print('AIC: ', result.aic)
    print('BIC: ', result.bic)
    print('FPE: ', result.fpe)
    print('HQIC: ', result.hqic)
    print('\n')

In [ ]:
#alternatively choose order (p) of VAR model using below
x = model.select_order(maxlags=4)
x.summary()

In [ ]:
#Train the VAR model of selected order(p) - here picked p=2
model_fitted = model.fit(2)
model_fitted.summary()

In [ ]:
#visualise the impact of changes in one variable on the others at different horizons using impulse responses plot - check if responses to the shocks die down after how many lags.
#Impulse Response Analysis, where the response of one variable to a sudden but temporary change in another variable is analyzed.
irf = model_fitted.irf(10)
irf.plot()
plt.show()

In [ ]:
#FEVD - Forecast Error Variance Decomposition
#shows how much of the future uncertainty of one time series is due to future shocks into the other time series
fevd = model_fitted.fevd(10)

fevd.summary()

In [ ]:
model_fitted.fevd(10).plot()

In [ ]:
#check for Serial Correlation of Residuals(Errors) using Durbin Watson (DW) Statistic
#check for serial correlation is to ensure that the model is sufficient to explain the variances and patterns in the time series.
#value vary between 0 and 4. Closer to 2, then there is no significant serial correlation; Closer to 0, there is a positive serial correlation; Closer to 4, there is a negative serial correlation.
from statsmodels.stats.stattools import durbin_watson
out = durbin_watson(model_fitted.resid)

def adjust(val, length=6): return str(val).ljust(length)

for col, val in zip(series.columns, out):
    print(adjust(col), ':', round(val, 2))

In [ ]:
#Forecast using VAR model
#need to provide as many of the previous values as indicated by the lag order (2) used by the model
lag_order = model_fitted.k_ar
print(lag_order)

#input data for forecast
#forecast_input = df_differenced1.values[-lag_order:]
forecast_input = df_train1.values[-lag_order:]
forecast_input

In [ ]:
df_train1

In [ ]:
#forecast now
fc = model_fitted.forecast(y=forecast_input, steps=sp)
df_forecast = pd.DataFrame(fc, index=series.index[-sp:], columns=series.columns+'_1d')

df_forecast

In [ ]:
#now de-difference once to bring it back to its original scale (only for the columns that has been differenced)
def invert_transformation(df_train, df_differenced, df_forecast, second_diff=False):
    #revert back the differencing
    df_fc = df_forecast.copy()
    columns = df_differenced.columns
    for col in columns:
        #roll back 1st diff
        df_fc[str(col)+'_forecast'] = df_train[col].iloc[-1] + df_fc[str(col)+'_1d'].cumsum()
        #print(df_train[col].iloc[-1])
        print(df_fc[str(col)+'_1d'].cumsum())
    for col in ["1M USDCNY", "Import Price Index(HS2)"]:
        df_fc[str(col)+'_forecast'] = df_fc[str(col)+'_1d']
    return df_fc

In [ ]:
df_results = invert_transformation(df_train, df_differenced, df_forecast, second_diff=False)
df_results.loc[:, ['1M USDCNY_forecast', 'FRD_forecast', 'CRD_forecast', 'PRD_forecast', 'RJ/CRB Index_forecast', 'Import Price Index(HS2)_forecast']] 

In [ ]:
#Plot Forecast values vs Actual values
fig, axes = plt.subplots(nrows=int(len(series.columns)/2), ncols=2, dpi=150, figsize=(10,10))
for i, (col, ax) in enumerate(zip(series.columns, axes.flatten())):
    df_results[col+'_forecast'].plot(legend=True, ax=ax).autoscale(axis='x',tight=True)
    df_test[col][-sp:].plot(legend=True, ax=ax);
    ax.set_title(col + ": Forecast vs Actuals")
    #ax.xaxis.set_ticks_position('none')
    #ax.yaxis.set_ticks_position('none')
    ax.spines["top"].set_alpha(0)
    ax.tick_params(labelsize=6)

plt.tight_layout();

In [ ]:
combine =[]
#combine the forecast and actual values then calculate the accuracy
combine = pd.concat([df_test['1M USDCNY'], df_results['1M USDCNY_forecast']], axis=1)
combine

In [ ]:
combine['accuracy'] = round(combine.apply(lambda row: row['1M USDCNY'] / row['1M USDCNY_forecast'] *100, axis = 1), 4)
combine['accuracy'] = pd.Series(["{0:.2f}%".format(val) for val in combine['accuracy']],index = combine.index)
#combine = combine.assign(day_of_week = lambda x: x.index.day_name())
combine = combine.round(decimals=4)
combine = combine.reset_index()
combine

In [ ]:
from statsmodels.tsa.stattools import acf

def forecast_accuracy(forecast, actual):
    mape = np.mean(np.abs(forecast - actual)/np.abs(actual))  # MAPE
    me = np.mean(forecast - actual)             # ME
    mae = np.mean(np.abs(forecast - actual))    # MAE
    mpe = np.mean((forecast - actual)/actual)   # MPE
    rmse = np.mean((forecast - actual)**2)**.5  # RMSE
    corr = np.corrcoef(forecast, actual)[0,1]   # corr
    mins = np.amin(np.hstack([forecast[:,None], actual[:,None]]), axis=1)
    maxs = np.amax(np.hstack([forecast[:,None], actual[:,None]]), axis=1)
    minmax = 1 - np.mean(mins/maxs)             # minmax
    return({'mape':mape, 'me':me, 'mae': mae, 
            'mpe': mpe, 'rmse':rmse, 'corr':corr, 'minmax':minmax})

print('Forecast Accuracy of: 1M USDCNY')
accuracy_prod = forecast_accuracy(df_results['1M USDCNY_forecast'], df_test['1M USDCNY'])
for k, v in accuracy_prod.items():
    print(adjust(k), ': ', round(v,4))

print('\nForecast Accuracy of: FRD')
accuracy_prod = forecast_accuracy(df_results['FRD_forecast'], df_test['FRD'])
for k, v in accuracy_prod.items():
    print(adjust(k), ': ', round(v,4))

print('\nForecast Accuracy of: CRD')
accuracy_prod = forecast_accuracy(df_results['CRD_forecast'], df_test['CRD'])
for k, v in accuracy_prod.items():
    print(adjust(k), ': ', round(v,4))

print('\nForecast Accuracy of: PRD')
accuracy_prod = forecast_accuracy(df_results['PRD_forecast'], df_test['PRD'])
for k, v in accuracy_prod.items():
    print(adjust(k), ': ', round(v,4))

print('\nForecast Accuracy of: RJ/CRB Index')
accuracy_prod = forecast_accuracy(df_results['RJ/CRB Index_forecast'], df_test['RJ/CRB Index'])
for k, v in accuracy_prod.items():
    print(adjust(k), ': ', round(v,4))

print('\nForecast Accuracy of: Import Price Index(HS2)')
accuracy_prod = forecast_accuracy(df_results['Import Price Index(HS2)_forecast'], df_test['Import Price Index(HS2)'])
for k, v in accuracy_prod.items():
    print(adjust(k), ': ', round(v,4))


In [ ]:
#####Now don't split the dataset, just build the model and train, then make predictions!!!##############
#select the best order (p) of VAR model
#pick the order that gives a model with least AIC, BIC, FPE, HQIC
model = VAR(df_differenced1)

for i in [0,1,2,3,4]:
    result = model.fit(i)
    print('Lag Order= ', i)
    print('AIC: ', result.aic)
    print('BIC: ', result.bic)
    print('FPE: ', result.fpe)
    print('HQIC: ', result.hqic)
    print('\n')

In [ ]:
#alternatively choose order (p=2) of VAR model using below
x = model.select_order(maxlags=4)
x.summary()

In [ ]:
#Train the VAR model of selected order(p) - here picked p=2
model_fitted = model.fit(2)
model_fitted.summary()

In [ ]:
#check for Serial Correlation of Residuals(Errors) using Durbin Watson (DW) Statistic
#check for serial correlation is to ensure that the model is sufficient to explain the variances and patterns in the time series.
#value vary between 0 and 4. Closer to 2, then there is no significant serial correlation; Closer to 0, there is a positive serial correlation; Closer to 4, there is a negative serial correlation.
from statsmodels.stats.stattools import durbin_watson
out = durbin_watson(model_fitted.resid)

def adjust(val, length=6): return str(val).ljust(length)

for col, val in zip(series.columns, out):
    print(adjust(col), ':', round(val, 2))

In [ ]:
#Forecast using VAR model
#need to provide as many of the previous values as indicated by the lag order (2) used by the model
lag_order = model_fitted.k_ar
print(lag_order)

#input data for forecast
forecast_input = df_differenced1.values[-lag_order:]
forecast_input

In [ ]:
#forecast now for 12 periods/months ahead, this is chosen by end users
fc = model_fitted.forecast(y=forecast_input, steps=12)
#df_forecast = pd.DataFrame(fc, index=series.index[-12:], columns=series.columns+'_1d')

df_forecast = pd.DataFrame(fc, columns=series.columns+'_1d')

df_forecast

In [ ]:
df_results = invert_transformation(df_train, df_differenced, df_forecast, second_diff=False)
df_results.loc[:, ['1M USDCNY_forecast', 'FRD_forecast', 'CRD_forecast', 'PRD_forecast', 'RJ/CRB Index_forecast', 'Import Price Index(HS2)_forecast']] 

In [ ]:
#extract model coefficients in preparation for monte carlo estimation
#visualize the model 
model_fitted.plot()

In [ ]:
#Model predictor variables forecast - 12 periods/months
model_fitted.plot_forecast(12)

In [ ]:
df_results.loc[:, ['RJ/CRB Index_forecast']].plot()
#'FRD_forecast', 'CRD_forecast', 'PRD_forecast', 'RJ/CRB Index_forecast', 'Import Price Index(HS2)_forecast']].plot()

In [ ]:
print('%s' %(model_fitted.params['1M USDCNY']))

In [ ]:
print(model_fitted.params.shape)

In [ ]:
a = model_fitted.params['1M USDCNY']
print(a.dtype)
a

In [ ]:
df_differenced1

In [ ]:
#prove that VAR model calculation for 2022-04-01 "1M USDCNY_forecast" is correct by using coefficients of constant and predictor variables - refer to Out[42]
b = np.array([1, 6.3462, -0.20257, 0.00, -1.70, 26.11, 113.1, 6.3160, -0.24657, -0.40, -0.60, 13.95, 111.9])

c = sum (a.values * b)

c

In [ ]:
#prove that VAR model calculation for 2022-05-01 "1M USDCNY_forecast" is correct by using coefficients of constant and predictor variables - refer to Out[42]
b1 = np.array([1, 6.343373, -0.158190, -0.345936, -0.031744, 1.488723, 113.886216, 6.3462, -0.20257, 0.00, -1.70, 26.11, 113.1])

c1 = sum (a.values * b1)

c1

In [ ]:
a1 = model_fitted.params['FRD']
print(a1.dtype)
a1

In [ ]:
#prove that VAR model calculation for 2022-04-01 "FRD_forecast" is correct by using coefficients of constant and predictor variables - refer to Out[42]
b = np.array([1, 6.3462, -0.20257, 0.00, -1.70, 26.11, 113.1, 6.3160, -0.24657, -0.40, -0.60, 13.95, 111.9])

c = sum (a1.values * b)

c

In [ ]:
#prove that VAR model calculation for 2022-05-01 "FRD_forecast" is correct by using coefficients of constant and predictor variables - refer to Out[42]
b1 = np.array([1, 6.343373, -0.158190, -0.345936, -0.031744, 1.488723, 113.886216, 6.3462, -0.20257, 0.00, -1.70, 26.11, 113.1])

c1 = sum (a1.values * b1)

c1

In [ ]:
# transform a time series dataset into a supervised learning dataset
def series_to_supervised(data, n_in=1, dropnan=True):
    """
    Frame a time series as a supervised learning dataset.
    Arguments:
        data: Sequence of observations as a list or NumPy array.
        n_in: Number of lag observations as input (X).
        n_out: Number of observations as output (y).
        dropnan: Boolean whether or not to drop rows with NaN values.
    Returns:
        Pandas DataFrame of series framed for supervised learning.
    """
    n_vars = 1 if type(data) is list else data.shape[1]
    df = pd.DataFrame(data)
    cols = list()
    # input sequence (t-n, ... t-1)
    for i in range(n_in, 0, -1):
        cols.append(df.shift(i))
    # forecast sequence (t, t+1, ... t+n)
    #for i in range(0, n_out):
        #cols.append(df.shift(-i))
    # put it all together
    agg = pd.concat(cols, axis=1)
    # drop rows with NaN values
    if dropnan:
        agg.dropna(inplace=True)
    return agg.values

In [ ]:
# transform the time series data into supervised learning
train = series_to_supervised(df_differenced1, n_in=2)

train

In [ ]:
train = pd.DataFrame(train)

In [ ]:
rows = len(train)

matrix = np.ones((rows, 1))

matrix = pd.DataFrame(matrix)

print(matrix.shape)

In [ ]:
train1 = pd.concat([matrix, train], axis=1, join='inner')

train1 = train1.to_numpy()

print(train1.shape)

In [ ]:
#### start derive Ut for all features

In [ ]:
#need to produce Ut for all features/variables using model parameters at each historical data poitns
matrix = model_fitted.params

matrix.values

In [ ]:
#re-order data for "1M USDCNY_forecast" past value predictions
a1 = np.array(matrix.values)

idx = [0, 7, 8, 9, 10, 11, 12, 1, 2, 3, 4, 5, 6]

#b = a1[:,idx]

b = a1[idx,:]

b

In [ ]:
d = np.dot(train1, b)

d

In [ ]:
train1

In [ ]:
matrix

In [ ]:
print(df_differenced1.shape)
print(d.shape)

In [ ]:
#derive Ut residual values at all historical data points for Monte Carlo simulation stage afterwards 
d1 = df_differenced1[2:] - d

d1

In [ ]:
d1.to_csv('Ut.csv')

In [ ]:
# Print datatypes
print(d1.dtypes)

#data analysis - describe all columns
d1.describe(include='all')

In [ ]:
d1.values

In [ ]:
#select Ut values randomly from historical data points
import random

#r = random.choice(d1.values)

#r

In [ ]:
#predict n future values of all features, simulate m times for all features
m = 1000
n = 6

p1 = np.zeros((n, m))
p2 = np.zeros((n, m)) 
p3 = np.zeros((n, m)) 
p4 = np.zeros((n, m)) 
p5 = np.zeros((n, m)) 
p6 = np.zeros((n, m)) 

In [ ]:
array = df_differenced1.values[-3:]

# transform a time series dataset into a supervised learning dataset
def series_to_supervised1(data, n_in=1, dropnan=True):
    """
    Frame a time series as a supervised learning dataset.
    Arguments:
        data: Sequence of observations as a list or NumPy array.
        n_in: Number of lag observations as input (X).
        n_out: Number of observations as output (y).
        dropnan: Boolean whether or not to drop rows with NaN values.
    Returns:
        Pandas DataFrame of series framed for supervised learning.
    """
    n_vars = 1 if type(data) is list else data.shape[1]
    df = pd.DataFrame(data)
    cols = list()
    # input sequence (t-n, ... t-1)
    for i in range(n_in,0,-1):
        cols.append(df.shift(i))
    # forecast sequence (t, t+1, ... t+n)
    #for i in range(0, n_out):
        #cols.append(df.shift(-i))
    # put it all together
    agg = pd.concat(cols, axis=1)
    # drop rows with NaN values
    if dropnan:
        agg.dropna(inplace=True)
    return agg.values


In [ ]:
# monte carlo simulation
m = 1000
n = 6

preds = []

for i in range(m):
    for j in range(n):
        print(j)
        # transform the time series data into supervised learning
        train = series_to_supervised1(array, n_in=2)
        train = pd.DataFrame(train)
        print(train)
        # add 1 in front of train sequence
        constant = np.ones((1, 1))
        constant = pd.DataFrame(constant)
        train1 = pd.concat([constant, train], axis=1, join='inner')
        train1 = train1.to_numpy()
        #print(train1)
        #re-order data for "1M USDCNY_forecast" past value predictions
        a1 = np.array(matrix.values)
        idx = [0, 7, 8, 9, 10, 11, 12, 1, 2, 3, 4, 5, 6]
        b = a1[idx,:]
        #print(a1)
        #derive one-step future T prediction
        d = np.dot(train1, b)
        #print(d)
        #add random Ut data to derive Ft
        r = random.choice(d1.values)
        combine = d + r
        print(combine)
        array = np.append(train, combine)
        notused = np.zeros((1,6))
        array = np.append(array, notused)
        array = array.reshape(-6,6)
        array = np.delete(array, 0, 0)
        print(array)
        
        #assign predicted values to all features n*m times
        p1[j][i] = combine[0][0]
        p2[j][i] = combine[0][1]
        p3[j][i] = combine[0][2]
        p4[j][i] = combine[0][3]
        p5[j][i] = combine[0][4]
        p6[j][i] = combine[0][5]
        
        #preds.append(combine)

In [ ]:
p1

In [ ]:
import seaborn as sns

sns.distplot(p1[0][:], color='blue')

In [ ]:
p1_new = p1.T

p1_new

In [ ]:
p1 = pd.DataFrame(p1)
p1.to_csv('param1.csv')

In [ ]:
p1_new = pd.DataFrame(p1_new)
p1_new.to_csv('param1_new.csv')

In [ ]:
p2 = pd.DataFrame(p2)
p2.to_csv('param2.csv')

p3 = pd.DataFrame(p3)
p3.to_csv('param3.csv')

p4 = pd.DataFrame(p4)
p4.to_csv('param4.csv')

p5 = pd.DataFrame(p5)
p5.to_csv('param5.csv')

p6 = pd.DataFrame(p6)
p6.to_csv('param6.csv')

In [ ]:
### using below VAR model embedded method can deduce Ut residual values even faster 
test =  model_fitted.resid

test

In [ ]:
###Need to find sorting points (分位点) for both key feature and rest of features, so to derive predictions for rest of features.
###key feature predictions will be given (manually entered by the end users) to determine sorting points at different future time points.

In [ ]:
###give key feature predictions for next 6 time points
list_usdcny = [6.44, 6.57, 6.78, 6.59, 6.48, 6.37]

list_usdcny[0]

In [ ]:
p1

In [ ]:
q1 = np.sort(p1, axis=1)
q2 = np.sort(p2, axis=1)
q3 = np.sort(p3, axis=1)
q4 = np.sort(p4, axis=1)
q5 = np.sort(p5, axis=1)
q6 = np.sort(p6, axis=1)

In [ ]:
q2

In [ ]:
fc =[]
for j in range(n):
    index = np.argmin(np.abs(np.array(q1[j][:])-list_usdcny[j]))
    #print(index)
    print(q1[j][index], q2[j][index], q3[j][index], q4[j][index], q5[j][index], q6[j][index])
    fc.append([q1[j][index], q2[j][index], q3[j][index], q4[j][index], q5[j][index], q6[j][index]])

In [ ]:
fc

In [ ]:
df_forecast = pd.DataFrame(fc, columns=series.columns+'_1d')

df_forecast

In [ ]:
#remember the forecast started from T, so need to use T-1 data even though Ut残差 calculated using 0 to T time points
series.iloc[-2]

In [ ]:
####invert the data for differenced ones####
#now de-difference once to bring it back to its original scale (only for the columns that has been differenced)
def invert_transformation(df_train, df_differenced, df_forecast, second_diff=False):
    #revert back the differencing
    df_fc = df_forecast.copy()
    columns = df_differenced.columns
    for col in columns:
        #roll back 1st diff
        df_fc[str(col)+'_forecast'] = df_train[col].iloc[-2] + df_fc[str(col)+'_1d'].cumsum()
        print(df_fc[str(col)+'_1d'].cumsum())
    for col in ["1M USDCNY", "Import Price Index(HS2)"]:
        df_fc[str(col)+'_forecast'] = df_fc[str(col)+'_1d']
    return df_fc

df_results = invert_transformation(series, df_differenced, df_forecast, second_diff=False)
df_results.loc[:, ['1M USDCNY_forecast', 'FRD_forecast', 'CRD_forecast', 'PRD_forecast', 'RJ/CRB Index_forecast', 'Import Price Index(HS2)_forecast']] 